In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", None)

ds = fetch_ucirepo(id=296)
df = ds.data.features.copy()
df["readmitted"] = ds.data.targets["readmitted"]

# Binary target: 1 = readmitted within 30 days (the costly event), 0 = not
df["target"] = (df["readmitted"] == "<30").astype(int)

n = len(df)
pos = df["target"].sum()
print(f"Encounters: {n:,}")
print(f"Positive (readmit <30d): {pos:,}  ({pos/n:.1%})")
print(f"Negative:                {n-pos:,}  ({1-pos/n:.1%})")
print(f"\nImbalance ratio: 1 positive for every {(n-pos)/pos:.1f} negatives")

In [ ]:
# The '?' trap: missing values are stored as the literal string "?",
# so pandas doesn't flag them as NaN. df.info() would look deceptively clean.
print("Columns with disguised missing values ('?'):\n")
q_counts = (df == "?").sum()
q_counts = q_counts[q_counts > 0].sort_values(ascending=False)
for col, cnt in q_counts.items():
    print(f"  {col:25s} {cnt:7,}  ({cnt/n:.1%})")

print(f"\nColumns affected: {len(q_counts)}")

In [ ]:
# Check for real NaN missing values (ucimlrepo may have already converted '?' -> NaN)
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print("Columns with missing values (NaN):\n")
for col, cnt in miss.items():
    print(f"  {col:25s} {cnt:7,}  ({cnt/n:.1%})")
print(f"\nColumns affected: {len(miss)}")

In [ ]:
# ID columns (identifiers, not features) and a look at the target-adjacent columns
print("Shape:", df.shape)
print("\nID-like columns present:",
      [c for c in ["encounter_id", "patient_nbr"] if c in df.columns])

# The repeat-patient trap: same patient can appear many times
if "patient_nbr" in df.columns:
    dup = df["patient_nbr"].duplicated().sum()
    print(f"Duplicate patient encounters: {dup:,} "
          f"({dup/n:.1%} of rows are repeat visits)")
    print(f"Unique patients: {df['patient_nbr'].nunique():,}")

In [ ]:
# ucimlrepo often separates IDs into ds.data.ids
print("Available data attributes:")
for attr in ["features", "targets", "ids"]:
    obj = getattr(ds.data, attr, None)
    if obj is not None:
        print(f"  ds.data.{attr}: shape {obj.shape}, columns {list(obj.columns)[:5]}")
    else:
        print(f"  ds.data.{attr}: None")

In [ ]:
ids = ds.data.ids
df["patient_nbr"] = ids["patient_nbr"].values

dup = df["patient_nbr"].duplicated().sum()
print(f"Total encounters:  {len(df):,}")
print(f"Unique patients:   {df['patient_nbr'].nunique():,}")
print(f"Repeat encounters: {dup:,}  ({dup/len(df):.1%} of rows are not a patient's first visit)")

# How many patients have multiple visits?
visits = df["patient_nbr"].value_counts()
multi = (visits > 1).sum()
print(f"\nPatients with >1 visit: {multi:,}")
print(f"Max visits by one patient: {visits.max()}")

In [ ]:
import sys
sys.path.insert(0, "../src")
from readmit.clean import load_clean

clean_df = load_clean()
print("Cleaned shape:", clean_df.shape)
print("Target rate:", f"{clean_df['target'].mean():.1%}")
print("Any missing left:", clean_df.isna().sum().sum())
print("\nColumns:", list(clean_df.columns))

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from readmit.clean import load_clean

sns.set_theme(style="whitegrid")
df = load_clean()
BASE_RATE = df["target"].mean()
print(f"Rows: {len(df):,}   Base readmission rate: {BASE_RATE:.1%}")

In [ ]:
# Hypothesis 1: more prior inpatient visits -> higher readmission rate
feat = "number_inpatient"
tab = (df.groupby(feat)["target"]
         .agg(["mean", "count"])
         .rename(columns={"mean": "readmit_rate", "count": "n"}))
tab = tab[tab["n"] >= 30]  # ignore tiny bins (unreliable rates)
print(tab.head(12))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tab.index, tab["readmit_rate"], color="steelblue")
ax.axhline(BASE_RATE, color="crimson", linestyle="--", label=f"base rate {BASE_RATE:.1%}")
ax.set_xlabel("prior inpatient visits (past year)")
ax.set_ylabel("readmission rate")
ax.set_title("Readmission rate rises sharply with prior inpatient visits")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Hypothesis 2: more diagnoses -> higher readmission rate
feat = "number_diagnoses"
tab = (df.groupby(feat)["target"]
         .agg(["mean", "count"])
         .rename(columns={"mean": "readmit_rate", "count": "n"}))
tab = tab[tab["n"] >= 30]
print(tab)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tab.index, tab["readmit_rate"], color="seagreen")
ax.axhline(BASE_RATE, color="crimson", linestyle="--", label=f"base rate {BASE_RATE:.1%}")
ax.set_xlabel("number of diagnoses")
ax.set_ylabel("readmission rate")
ax.set_title("Readmission rate vs. number of diagnoses")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Hypothesis 3: discharge disposition (categorical) vs readmission
feat = "discharge_disposition_id"
tab = (df.groupby(feat)["target"]
         .agg(["mean", "count"])
         .rename(columns={"mean": "readmit_rate", "count": "n"}))
tab = tab[tab["n"] >= 100].sort_values("readmit_rate", ascending=False)
print(tab.head(12))

fig, ax = plt.subplots(figsize=(8, 4))
top = tab.head(10)
ax.barh(top.index.astype(str), top["readmit_rate"], color="darkorange")
ax.axvline(BASE_RATE, color="crimson", linestyle="--", label=f"base rate {BASE_RATE:.1%}")
ax.set_xlabel("readmission rate")
ax.set_ylabel("discharge_disposition_id")
ax.set_title("Readmission rate by discharge disposition (top 10)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Deceased/hospice dispositions can't be readmitted - must be removed
expired_hospice = [11, 13, 14, 19, 20, 21]  # expired or hospice per mapping
present = df["discharge_disposition_id"].isin(expired_hospice).sum()
print(f"Rows with expired/hospice disposition: {present:,} ({present/len(df):.1%})")
print(df[df["discharge_disposition_id"].isin(expired_hospice)]
      .groupby("discharge_disposition_id")["target"].agg(["mean","count"]))

In [ ]:
from importlib import reload
import readmit.clean as clean_mod
reload(clean_mod)

df2 = clean_mod.load_clean()
print("Rows before fix: 71,518")
print(f"Rows after removing expired/hospice: {len(df2):,}")
print(f"New base rate: {df2['target'].mean():.1%}")
# Confirm none remain
remaining = df2["discharge_disposition_id"].isin([11,13,14,19,20,21]).sum()
print(f"Expired/hospice rows remaining: {remaining}")